# 01 · Frozen DINOv3 global baseline (Kaggle GPU)

Notebook chạy B0 global + linear và B1 global + MLP trên split Phase 2. Kết quả chỉ là in-domain validation; không có source metadata để đánh giá unseen-source.

## Trước khi chạy

1. Chọn **GPU** trong Kaggle Notebook Settings.
2. Add Input là bộ dữ liệu Who Is AI có train/manifest.csv và train/images/.
3. Đảm bảo code repo mới nhất có trên GitHub hoặc được add vào Kaggle như input. Cell setup clone repo nếu chưa tìm thấy source local.
4. Trên Hugging Face, tự chấp nhận điều khoản và xin quyền với checkpoint DINOv3. Thêm Kaggle Secret có tên HF_TOKEN; notebook không in token.
5. Bật Internet trong Kaggle để cài Transformers và tải checkpoint nếu chưa được cache.

Các artifact được ghi vào /kaggle/working/phase3_outputs/. Sau khi chạy, dùng Save Version để lưu notebook output; cell cuối cũng đóng gói artifact thành ZIP.


In [1]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/AIVIETNAM-AIO-DinhBao/FaceTrace.git"
REPO_REF = "main"
REPO_DIR = Path("/kaggle/working/FaceTrace")

# If the repository is already attached as a Kaggle Input, set REPO_DIR to that path.
if not (REPO_DIR / "src").is_dir():
    attached_candidates = [
        Path("/kaggle/input/facetrace"),
        Path("/kaggle/input/face-trace"),
    ]
    attached_repo = next((p for p in attached_candidates if (p / "src").is_dir()), None)
    if attached_repo is not None:
        REPO_DIR = attached_repo
    else:
        REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)],
            check=True,
        )

sys.path.insert(0, str(REPO_DIR))
print("Repo:", REPO_DIR)


Cloning into '/kaggle/working/FaceTrace'...


Repo: /kaggle/working/FaceTrace


In [2]:
%pip install -q "transformers>=4.56.1,<5" "PyYAML>=6" "scikit-learn>=1.4"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 90.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.3 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [3]:
import torch
import transformers

if not torch.cuda.is_available():
    raise RuntimeError("GPU chưa được bật. Trong Kaggle chọn Settings → Accelerator → GPU rồi chạy lại.")

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)


GPU: Tesla T4
PyTorch: 2.10.0+cu128
Transformers: 4.57.6


In [4]:
from pathlib import Path

# Tự tìm manifest trong Kaggle Inputs; có thể gán DATA_ROOT thủ công nếu cấu trúc khác.
DATA_ROOT = None
if DATA_ROOT is None:
    input_root = Path("/kaggle/input")
    manifests = sorted(input_root.rglob("manifest.csv")) if input_root.exists() else []
    matches = [p for p in manifests if p.parent.name == "train" and (p.parent / "images").is_dir()]
    if not matches:
        raise FileNotFoundError(
            "Không tìm thấy train/manifest.csv cùng train/images/. "
            "Add Input bộ dữ liệu Who Is AI hoặc đặt DATA_ROOT tới thư mục có train/."
        )
    DATA_ROOT = matches[0].parent.parent
else:
    DATA_ROOT = Path(DATA_ROOT)

TRAIN_ROOT = DATA_ROOT / "train"
MANIFEST_PATH = TRAIN_ROOT / "manifest.csv"
assert MANIFEST_PATH.is_file(), MANIFEST_PATH
print("Dataset:", DATA_ROOT)
print("Train manifest:", MANIFEST_PATH)


Dataset: /kaggle/input/datasets/dinhbaobao/who-is-ai/data
Train manifest: /kaggle/input/datasets/dinhbaobao/who-is-ai/data/train/manifest.csv


In [5]:
import os

HF_TOKEN = os.environ.get("HF_TOKEN")
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = HF_TOKEN or UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

if not HF_TOKEN:
    raise RuntimeError(
        "Thêm Kaggle Secret tên HF_TOKEN sau khi tài khoản Hugging Face đã được cấp quyền model."
    )

from huggingface_hub import login
login(token=HF_TOKEN, add_to_git_credential=False)
del HF_TOKEN
print("Hugging Face authentication configured; token hidden.")


Hugging Face authentication configured; token hidden.


In [6]:
import importlib.util
import pandas as pd

split_spec = importlib.util.spec_from_file_location(
    "phase2_split_generator", REPO_DIR / "scripts" / "create_splits.py"
)
split_module = importlib.util.module_from_spec(split_spec)
split_spec.loader.exec_module(split_module)

manifest_rows = split_module.read_manifest(MANIFEST_PATH)
train_rows, val_rows = split_module.create_splits(
    manifest_rows,
    validation_fraction=0.20,
    seed=42,
    train_images_root=TRAIN_ROOT,
    image_path_base=REPO_DIR,
)

SPLIT_DIR = Path("/kaggle/working/phase3_splits")
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_CSV = SPLIT_DIR / "train.csv"
VAL_CSV = SPLIT_DIR / "val.csv"
split_module.write_csv(TRAIN_CSV, train_rows)
split_module.write_csv(VAL_CSV, val_rows)

counts = pd.DataFrame(train_rows + val_rows).groupby(
    ["split", "label", "class_name"], dropna=False
).size().rename("n_images").reset_index()
display(counts)
assert len(train_rows) == 1600 and len(val_rows) == 400


,split,label,class_name,n_images
0,train,0,Real,800
1,train,1,Fake,800
2,val,0,Real,200
3,val,1,Fake,200


In [7]:
import yaml

BASE_CONFIG = yaml.safe_load((REPO_DIR / "configs" / "baseline.yaml").read_text(encoding="utf-8"))
BASE_CONFIG["data"]["train_manifest"] = str(TRAIN_CSV)
BASE_CONFIG["data"]["validation_manifest"] = str(VAL_CSV)
BASE_CONFIG["data"]["image_path_base"] = str(REPO_DIR)
BASE_CONFIG["experiment"]["output_dir"] = "/kaggle/working/phase3_outputs"
BASE_CONFIG["training"]["device"] = "cuda"

RUN_CONFIG = Path("/kaggle/working/phase3_config.yaml")
RUN_CONFIG.write_text(yaml.safe_dump(BASE_CONFIG, sort_keys=False), encoding="utf-8")
print(RUN_CONFIG.read_text(encoding="utf-8"))


experiment:
  name: dinov3_global_baseline
  seed: 42
  output_dir: /kaggle/working/phase3_outputs
data:
  train_manifest: /kaggle/working/phase3_splits/train.csv
  validation_manifest: /kaggle/working/phase3_splits/val.csv
  image_path_base: /kaggle/working/FaceTrace
  image_column: image_path
  label_column: label
  source_column: source
  image_size: 224
  batch_size: 32
  num_workers: 4
model:
  backbone: facebook/dinov3-vitb16-pretrain-lvd1689m
  pretrained: true
  freeze_backbone: true
  revision: null
  feature: cls
  classifier: linear
  hidden_dim: 256
  dropout: 0.0
  num_classes: 2
training:
  epochs: 20
  learning_rate: 0.001
  weight_decay: 0.0001
  device: cuda
  mixed_precision: true
  pin_memory: true
evaluation:
  threshold: 0.5
  metrics:
  - auroc
  - balanced_accuracy
  - accuracy
  - f1



In [8]:
# Import and call the same entry point used by the repository modules.
import importlib.util

baseline_spec = importlib.util.spec_from_file_location(
    "phase3_baseline_runner", REPO_DIR / "scripts" / "train_baseline.py"
)
baseline_module = importlib.util.module_from_spec(baseline_spec)
baseline_spec.loader.exec_module(baseline_module)

sys.argv = ["train_baseline.py", "--config", str(RUN_CONFIG)]
baseline_module.main()


Python=3.12.13 torch=2.10.0+cu128 transformers=4.57.6 device=cuda


preprocessor_config.json:   0%|          | 0.00/585 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/744 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/343M [00:00<?, ?B/s]

/kaggle/working/FaceTrace/src/training/trainer.py:19: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


...

epoch=19 train_loss=0.00244 val_auroc=0.92377 val_bacc=0.83500


/kaggle/working/FaceTrace/src/training/trainer.py:19: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


train:   0%|          | 0/50 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f8be060f100>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
  Exception ignored in:  <function _MultiProcessingDataLoaderIter.__del__ at 0x7f8be060f100>
Traceback (most recent call last):
     File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
       self._shutdown_workers()  
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
^    ^if w.is_alive():^
^ ^ ^ ^ ^  ^ ^^^^^^^^^^^^^^^^

epoch=20 train_loss=0.00441 val_auroc=0.92358 val_bacc=0.83750
Selected head: mlp — validation AUROC 0.93228; run artifacts: /kaggle/working/phase3_outputs/dinov3_global_baseline_seed42


In [9]:
import json
import shutil
import pandas as pd

RUN_DIR = Path("/kaggle/working/phase3_outputs/dinov3_global_baseline_seed42")
results = json.loads((RUN_DIR / "results.json").read_text(encoding="utf-8"))
summary = []
for head, result in results.items():
    summary.append({"head": head, "best_epoch": result["best_epoch"], **result["metrics"]})
display(pd.DataFrame(summary))

archive = shutil.make_archive(
    "/kaggle/working/phase3_baseline_artifacts", "zip", root_dir=RUN_DIR
)
print("Artifact ZIP:", archive)
print("Selected head:", json.loads((RUN_DIR / "selected_head.json").read_text())["selected_head"])


,head,best_epoch,val_balanced_accuracy,val_accuracy,val_f1,val_auroc
0,linear,18,0.865,0.865,0.860104,0.926725
1,mlp,3,0.870,0.870,0.857143,0.932275


Artifact ZIP: /kaggle/working/phase3_baseline_artifacts.zip
Selected head: mlp
